In [ ]:
# Parameters (override from the scheduler, a pipeline activity or the Job Scheduler API)
version = ""  # empty = the version in Files/pipelines/hdx-scraper-ophi/LATEST
hdx_site = "stage"
key_vault_url = "https://chd-datasystems-kv.vault.azure.net/"
hdx_key_secret = "hdx-key"
user_agent = "hdx-scraper-ophi"
preprefix = ""

In [ ]:
import time
from pathlib import Path

root = Path("/lakehouse/default/Files/pipelines/hdx-scraper-ophi")
if not version:
    version = (root / "LATEST").read_text().strip()
release = root / version
wheel = next(release.glob("*.whl"))
start = time.monotonic()
%pip install -q --disable-pip-version-check -r {release}/requirements.txt
%pip install -q --disable-pip-version-check --no-deps {wheel}
print(f"Installed {wheel.name} in {time.monotonic() - start:.0f}s")

In [ ]:
import os
import runpy
import sys

if not key_vault_url:
    raise ValueError("Set key_vault_url to the Key Vault holding the HDX API key")
os.environ["HDX_KEY"] = notebookutils.credentials.getSecret(key_vault_url, hdx_key_secret)
os.environ["HDX_SITE"] = hdx_site
os.environ["USER_AGENT"] = user_agent
if preprefix:
    os.environ["PREPREFIX"] = preprefix
# The kernel's argv holds "-f <connection file>", which facade would parse as CLI arguments
sys.argv = ["hdx-scraper-ophi"]
runpy.run_module("hdx.scraper.ophi", run_name="__main__")